# COMP 365 — Build 6: From Raw Data to an AI-Ready Dataset

**Arthur Cox III**  
**Dataset:** scikit-learn Wine  
**Purpose:** Assess whether chemical measurements can support a limited educational classifier for the three wine cultivator classes. No model is trained.

**AI-use disclosure:** I used ChatGPT to help draft explanations, complete notebook code, and check the audit. The notebook output is generated by running the supplied code. Dataset provenance and permissions are marked for verification where uncertain.


## 0. Setup

Run the imports. Keep the random seed unchanged so your generated data can be reproduced.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.datasets import (
    load_iris,
    load_wine,
    load_breast_cancer,
    load_digits,
    make_classification,
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

## 1. Choose and load one built-in dataset

Choose **iris**, **wine**, **breast_cancer**, or **digits**. The loader returns a scikit-learn `Bunch`. Using `as_frame=True` gives you pandas objects.

In [ ]:
DATASET_LOADERS = {
    "iris": load_iris,
    "wine": load_wine,
    "breast_cancer": load_breast_cancer,
    "digits": load_digits,
}

# You may change this value.
DATASET_NAME = "wine"

loader = DATASET_LOADERS[DATASET_NAME]
bundle = loader(as_frame=True)
df = bundle.frame.copy()
target_col = bundle.target.name if bundle.target.name else "target"

print("Dataset:", DATASET_NAME)
print("Available keys:", list(bundle.keys()))
print("DataFrame shape:", df.shape)
display(df.head())

In [ ]:
print(bundle.DESCR[:1600])

### Dataset purpose and meaning

- **Proposed AI task:** Educational classification of a wine sample into one of the three source cultivator classes using 13 chemical measurements.
- **What one row represents:** One analyzed wine sample from the dataset's Italian growing region.
- **Target column and meaning:** `target`; codes 0, 1, and 2 identify the three cultivator classes. The loader does not provide cultivar names.
- **Source information found in `DESCR`:** A copy of UCI's Wine recognition dataset, donated in July 1988. It describes 178 samples, 13 chemical measurements, wines from one Italian region and three cultivators, and no missing attributes.
- **One fact that is still unclear:** Collection and sampling protocol, present-day representativeness, and permission for a proposed operational use **need verification**.


## 2. Profile the dataset

Run repeatable checks before making a model or cleaning anything.

In [ ]:
def profile_dataframe(frame, target):
    report = {
        "rows": frame.shape[0],
        "columns": frame.shape[1],
        "missing_cells": int(frame.isna().sum().sum()),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
        "numeric_columns": int(frame.select_dtypes(include=np.number).shape[1]),
        "non_numeric_columns": int(frame.select_dtypes(exclude=np.number).shape[1]),
        "target_unique_values": int(frame[target].nunique(dropna=False)),
    }
    return pd.Series(report, name="value")

display(profile_dataframe(df, target_col).to_frame())

print("\nColumn types")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values by column")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("\nTarget distribution")
display(df[target_col].value_counts(dropna=False).sort_index().to_frame("count"))

print("\nNumeric summary")
display(df.describe().T)

### Three profiling findings

1. There are **178 rows and 14 columns**: 13 numeric chemical measurements plus the numeric target.
2. The provided file has **0 missing cells and 0 exact duplicate rows**. This does not establish that measurements or labels are correct.
3. Classes 0, 1, and 2 have **59, 71, and 48 samples**, respectively. Class 2 is the smallest (about 27% of rows).

**Biggest quality risk visible so far:** Uneven and fairly small class counts could make a performance estimate for class 2 less stable than a pooled estimate.

**Important limitation:** Summary commands cannot reveal mislabeled samples, incorrect lab measurements, or whether the collection reflects wines encountered today.


## 3. Create two evidence-based visuals

The first plot checks target balance. The second checks how one numeric feature differs across target classes.

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x=target_col, color="#2455D9")
plt.title(f"Target distribution: {DATASET_NAME}")
plt.xlabel("Target class")
plt.ylabel("Number of rows")
plt.tight_layout()
plt.show()

**Interpretation of Plot 1:** Class 1 has 71 samples, class 0 has 59, and class 2 has 48. The imbalance is moderate, but a model could obtain a misleading overall score if one class is harder to identify. Evaluate performance separately for all three classes; the chart says nothing about label accuracy or population coverage.

In [ ]:
numeric_features = [c for c in df.select_dtypes(include=np.number).columns if c != target_col]

# You may choose a different numeric feature.
FEATURE_TO_PLOT = "alcohol"

plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x=target_col, y=FEATURE_TO_PLOT, color="#A78BFA")
plt.title(f"{FEATURE_TO_PLOT} by target class")
plt.xlabel("Target class")
plt.ylabel(FEATURE_TO_PLOT)
plt.tight_layout()
plt.show()

**Interpretation of Plot 2:** Alcohol measurements for class 0 have the highest median (about 13.75), class 1 the lowest (about 12.29), and class 2 lies between (about 13.17). Each class has a spread, their ranges overlap, and points beyond the boxplot whiskers should be checked rather than automatically called errors. Alcohol alone cannot perfectly identify the class; this plot does not prove causation, generalization, or measurement correctness.

## 4. Create a reproducible synthetic dataset

This dataset is designed for a controlled classification experiment. It is not evidence about real people or real events.

In [ ]:
X, y = make_classification(
    n_samples=250,
    n_features=6,
    n_informative=3,
    n_redundant=1,
    n_repeated=0,
    n_classes=2,
    weights=[0.75, 0.25],
    class_sep=1.0,
    flip_y=0.02,
    random_state=RANDOM_STATE,
)

feature_names = [f"feature_{i}" for i in range(X.shape[1])]
synthetic_df = pd.DataFrame(X, columns=feature_names)
synthetic_df["target"] = y
synthetic_df.insert(0, "generated_id", [f"S{i:03d}" for i in range(len(synthetic_df))])

display(synthetic_df.head())
display(synthetic_df["target"].value_counts().sort_index().to_frame("count"))

### Explain the generation choices

- **Why use `random_state`?** The fixed seed 42 makes the generated features, labels, and later audit reproducible.
- **What does `weights=[0.75, 0.25]` control?** It asks for approximately 75% class 0 and 25% class 1 before the 2% random label flipping; this run produced 187 and 63.
- **Why must we not present this as real-world evidence?** `make_classification` creates artificial patterns by rule; it does not represent observed people, farms, or transactions.


## 5. Break the synthetic dataset on purpose

The cell below adds four quality risks: missing feature values, duplicates, an invalid label, and an extreme feature value.

In [ ]:
broken_df = synthetic_df.copy()

# Missing feature values
broken_df.loc[0:9, "feature_0"] = np.nan

# Invalid target label: the declared target contract allows only 0 and 1
broken_df.loc[10, "target"] = 2

# Extreme value that needs investigation
broken_df.loc[11, "feature_1"] = 999.0

# Exact duplicates
broken_df = pd.concat([broken_df, broken_df.iloc[:3]], ignore_index=True)

display(broken_df.head(14))

In [ ]:
def audit_synthetic(frame):
    allowed_labels = {0, 1}
    observed_labels = set(frame["target"].dropna().unique())
    q1 = frame["feature_1"].quantile(0.25)
    q3 = frame["feature_1"].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return pd.Series({
        "rows": len(frame),
        "missing_cells": int(frame.isna().sum().sum()),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
        "invalid_target_rows": int((~frame["target"].isin(allowed_labels)).sum()),
        "feature_1_outlier_flags": int(((frame["feature_1"] < lower) | (frame["feature_1"] > upper)).sum()),
        "observed_labels": sorted(observed_labels),
    }, name="audit")

display(audit_synthetic(broken_df).to_frame())

### Audit interpretation

- **Which failures were detected?** The broken file has 253 rows, 13 missing cells (10 original missing values plus three copied in duplicate rows), 3 exact duplicate rows, 1 invalid target value `2`, and 5 IQR outlier flags in `feature_1`, including the inserted `999`.
- **Which result needs human investigation rather than automatic deletion?** The `999` and the other IQR flags. An extreme flag identifies a value for review; an unusual value is not always incorrect. We know `999` was deliberately inserted here, but a real dataset would require source verification.
- **What important problem could still remain undetected?** Plausible but incorrect labels or biased sampling would pass these simple checks.


## 6. Repair what you can justify

This repair removes exact duplicates, removes the row with an impossible target label, and imputes missing feature values using the median. It **flags** extreme values instead of silently deleting them.

In [ ]:
repaired_df = broken_df.copy()

# 1. Remove exact duplicate rows.
repaired_df = repaired_df.drop_duplicates().copy()

# 2. Remove rows whose target violates the declared {0, 1} contract.
# We cannot safely guess the correct label.
repaired_df = repaired_df[repaired_df["target"].isin([0, 1])].copy()

# 3. Impute missing numeric feature values with the column median.
feature_cols = [c for c in repaired_df.columns if c.startswith("feature_")]
repaired_df[feature_cols] = repaired_df[feature_cols].fillna(repaired_df[feature_cols].median())

# 4. Flag feature_1 outliers for review. Do not delete them automatically.
q1 = repaired_df["feature_1"].quantile(0.25)
q3 = repaired_df["feature_1"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
repaired_df["feature_1_review_flag"] = (
    (repaired_df["feature_1"] < lower) | (repaired_df["feature_1"] > upper)
)

display(audit_synthetic(repaired_df).to_frame())
display(repaired_df[repaired_df["feature_1_review_flag"]].head())

### Repair log

| Problem | Action | Why was this action justified? | What risk remains? |
|---|---|---|---|
| Exact duplicates | Remove 3 exact duplicate rows. | They were deliberately appended copies, including their IDs. | Legitimate repeated real-world measurements could look alike; investigate in a real dataset. |
| Invalid label | Remove the 1 row labeled `2`. | Synthetic target contract permits only 0 and 1; the correct label cannot be inferred. | A legitimate but wrong 0/1 label would pass. |
| Missing feature values | Fill remaining 10 missing `feature_0` values with the column median. | Retains rows without inventing extreme measurements; this is a demonstration repair. | Imputation hides variation and can bias results; in modeling, fit the median only on training data. |
| Extreme feature value | Keep observations and add `feature_1_review_flag`; 5 rows are flagged. | IQR flags need investigation; silently deleting them would be unjustified. | Known inserted `999` remains in the CSV pending review and must not be treated as clean for training. |


## 7. Complete the provenance card

In [ ]:
provenance_card = {
    "dataset_name": DATASET_NAME,
    "loader": f"sklearn.datasets.{loader.__name__}",
    "scikit_learn_version": sklearn.__version__,
    "access_date": pd.Timestamp.utcnow().date().isoformat(),
    "row_meaning": "One chemically analyzed wine sample from one Italian region",
    "target_meaning": "Codes 0, 1, 2 denote the three cultivator classes; names are not supplied",
    "source_summary": "Copy of UCI Wine recognition data; 178 wines, 13 chemical features, three cultivators in one Italian region; donor Michael Marshall, July 1988; original owners listed as Forina et al.",
    "transformations": [
        "Loaded with as_frame=True",
        "No changes made before profiling",
    ],
    "permission_status": "needs verification for any use beyond this educational exercise",
    "known_limitations": [
        "One Italian region and three cultivators; broader representativeness unknown",
        "Historical 1988 donation; collection details and modern applicability need verification",
        "Cultivator names and permission for operational reuse need verification"
    ],
}

provenance_card

## 8. Readiness decision

- **Specific proposed AI task:** Classify the three source cultivators from the 13 measured chemical features in a classroom prototype.
- **Decision:** **RESTRICT** — appropriate to explore as a labeled educational dataset, not approved for real-world wine origin decisions.
- **Evidence 1:** The supplied 178-row table has no missing cells and no exact duplicate rows in this profiling run.
- **Evidence 2:** The three labels have 59, 71, and 48 examples; class 2 is smallest, so aggregate accuracy alone would hide class-specific weakness.
- **Evidence 3:** Alcohol distributions differ by class but overlap, so at least one feature contains information without being a decisive standalone rule. The loader describes one Italian region and three cultivators, which limits scope.
- **Remaining risk:** Unknown sampling details, label verification, current representativeness, and permissions for deployment **need verification**.
- **What must happen next:** Confirm provenance and permission with the data owner; obtain new representative samples and independently validate labels before considering operational use.


## 9. Export and final check

The CSV contains your repaired synthetic dataset and the review flag. Your completed notebook contains the built-in dataset profile, visuals, provenance card, and readiness decision.

In [ ]:
OUTPUT_FILE = "build6_audited_dataset.csv"
repaired_df.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Rows exported:", len(repaired_df))
print("Missing cells:", int(repaired_df.isna().sum().sum()))
print("Exact duplicates:", int(repaired_df.duplicated().sum()))
print("Invalid targets:", int((~repaired_df["target"].isin([0, 1])).sum()))

## Submission checklist

- [x] All code cells run from top to bottom.
- [x] Three profiling findings and one major risk are documented.
- [x] Both plots have written interpretations.
- [x] Synthetic design and its limitations are explained.
- [x] Missing values, duplicates, invalid labels, and extremes are audited.
- [x] The repair log and provenance card are complete.
- [x] A RESTRICT decision has three pieces of evidence, remaining risk, and next step.
- [x] Notebook and `build6_audited_dataset.csv` are included in the ZIP.
